# 01 · Pobranie tekstów

Pierwszy krok projektu: zebrać teksty i opisać je metadanymi. Pracujemy jak w notatniku z analizą sieci: każdy krok to **prompt dla modelu → pusta komórka na kod → opis oczekiwanego wyniku**.

Efektem końcowym będą:
1. surowe teksty w folderze `data/raw/` (nie trafiają do repozytorium),
2. tabela `data/processed/teksty.csv` z tekstami i metadanymi, gotowa do dalszej analizy.

## Jak pracować z tym notatnikiem

1. Uzupełnij komórkę parametrów.
2. Skopiuj prompt z bieżącego kroku do czatu AI, np. Copilot Chat w Codespaces.
3. Wklej otrzymany kod do pustej komórki pod promptem.
4. Uruchom kod i porównaj wynik z sekcją **Po uruchomieniu powinieneś zobaczyć**.
5. Jeśli wynik się nie zgadza, popraw tylko bieżący krok. Dopiero potem idź dalej.

> **Zanim cokolwiek pobierzesz:** sprawdź regulamin serwisu i plik `robots.txt`, a w karcie projektu (pole 9) zapisz, co ustaliłeś. Nicki i nazwiska prywatnych osób zamieniamy na identyfikatory.

## Parametry

Jedyna komórka, którą zmieniasz ręcznie. Przykład poniżej to pierwszy tom „Lalki” z Wolnych Lektur (domena publiczna). Zamień go na swoje źródło.

In [ ]:
ZRODLO_NAZWA = "Wolne Lektury"
ZRODLO_ADRESY = [
    "https://wolnelektury.pl/media/book/txt/lalka-tom-pierwszy.txt",
]  # jeśli adres nie działa: strona książki na wolnelektury.pl → pobierz → TXT
DZIELO = "Lalka"            # czego dotyczą teksty
TYP_TEKSTU = "powieść"      # np. powieść, recenzja, komentarz, dialog
LIMIT_TESTOWY = 20          # najpierw mała próbka

---
## Krok 1A. Pobranie i zapis surowych plików

#### Cel i sens analityczny
Pobieramy teksty dokładnie w takiej postaci, w jakiej są w źródle, i zapisujemy je w `data/raw/`. Dzięki temu każdy dalszy krok można powtórzyć bez ponownego pobierania.

#### Prompt dla modelu

```text
Kontekst:
Pracuję w Jupyter Notebook w GitHub Codespaces (Python 3.12). Notatnik leży w folderze notebooks/, a dane zapisuję w ../data/raw/.

Wejście:
Lista adresów w zmiennej ZRODLO_ADRESY i limit LIMIT_TESTOWY.

Zadanie:
Pobierz zawartość każdego adresu (najwyżej LIMIT_TESTOWY adresów) biblioteką requests, z czytelnym nagłówkiem User-Agent i przerwą 1 sekundy między zapytaniami. Każdą odpowiedź zapisz jako osobny plik w ../data/raw/ z nazwą tekst_001.txt, tekst_002.txt itd., w kodowaniu UTF-8. Zachowaj listę pobranych plików w zmiennej pobrane.

Pokaż wynik:
- liczbę pobranych plików i liczbę błędów,
- dla każdego pliku: nazwę, adres i liczbę znaków,
- pierwsze 300 znaków pierwszego pliku.

Warunek poprawności:
Pliki nie mogą być puste, a polskie znaki muszą być czytelne.

Jeśli wystąpi błąd:
Wypisz, którego adresu dotyczy i czy to problem z adresem, połączeniem czy kodowaniem. Nie przerywaj pobierania pozostałych.

Nie rób jeszcze:
Nie czyść tekstu i nie twórz tabeli.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć
- Komunikat o liczbie pobranych plików.
- Listę plików z liczbą znaków.
- Początek pierwszego tekstu z poprawnymi polskimi znakami.

---
## Krok 1B. Tabela z tekstami i metadanymi

#### Cel i sens analityczny
Metadane zapisujemy **przy pobieraniu**, a nie każemy modelowi ich zgadywać. Data, źródło i autor przydadzą się w analizie: pozwolą porównywać grupy i sprawdzać model.

#### Prompt dla modelu

```text
Kontekst:
W ../data/raw/ mam pliki tekst_001.txt, tekst_002.txt itd. W zmiennych ZRODLO_NAZWA, DZIELO, TYP_TEKSTU mam opis źródła.

Wejście:
Pliki tekstowe z ../data/raw/ i lista adresów ZRODLO_ADRESY w tej samej kolejności.

Zadanie:
Zbuduj tabelę pandas o nazwie teksty z kolumnami: id (tekst_001...), tekst, dzielo, typ_tekstu, zrodlo, url, data_pobrania (dzisiejsza data), liczba_slow. Jeśli tekst zawiera nagłówek lub stopkę serwisu (np. informacje o licencji Wolnych Lektur), usuń je i napisz, jaką regułą to zrobiłeś.

Pokaż wynik:
- liczbę wierszy i nazwy kolumn,
- pierwsze 5 wierszy bez kolumny tekst,
- statystyki kolumny liczba_slow (min, mediana, max).

Warunek poprawności:
Każdy pobrany plik ma jeden wiersz, a kolumna tekst nie jest pusta.

Jeśli wystąpi błąd:
Wypisz, którego pliku dotyczy.

Nie rób jeszcze:
Nie zapisuj pliku CSV.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć
- Tabelę z jednym wierszem na tekst.
- Rozsądne liczby słów: brak tekstów pustych albo podejrzanie krótkich.

---
## Krok 1C. Pseudonimizacja i kontrola jakości

#### Cel i sens analityczny
Nicki i nazwiska prywatnych osób to dane osobowe. Zamieniamy je na identyfikatory, zanim cokolwiek opublikujemy. Przy okazji wyłapujemy duplikaty i teksty zbyt krótkie do analizy.

> Jeśli Twoje teksty nie mają autorów prywatnych (np. powieść), pomiń pseudonimizację i zrób tylko kontrolę jakości.

#### Prompt dla modelu

```text
Kontekst:
Mam tabelę teksty z kolumnami id, tekst, dzielo, typ_tekstu, zrodlo, url, data_pobrania, liczba_slow. Część tabel może mieć też kolumnę autor z nickami.

Wejście:
Tabela teksty.

Zadanie:
1. Jeśli istnieje kolumna autor, zamień każdą unikalną wartość na identyfikator autor_001, autor_002 itd. Słownik zamian zapisz do ../data/raw/slownik_autorow.csv (ten folder nie trafia do repozytorium).
2. Usuń duplikaty tekstów.
3. Oznacz w nowej kolumnie za_krotki teksty krótsze niż 20 słów.

Pokaż wynik:
- ile autorów zamieniono,
- ile duplikatów usunięto,
- ile tekstów jest za krótkich.

Warunek poprawności:
W tabeli teksty nie ma już oryginalnych nicków.

Jeśli wystąpi błąd:
Wyjaśnij, którego kroku dotyczy.

Nie rób jeszcze:
Nie usuwaj tekstów za krótkich, tylko je oznacz.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć
- Liczby zamian, duplikatów i tekstów za krótkich.
- Tabelę bez oryginalnych nicków.

---
## Krok 1D. Zapis tabeli do `data/processed/`

#### Prompt dla modelu

```text
Kontekst:
Mam oczyszczoną tabelę teksty.

Wejście:
Tabela teksty.

Zadanie:
Zapisz tabelę do ../data/processed/teksty.csv w kodowaniu UTF-8, bez indeksu. Wczytaj plik z powrotem i sprawdź, czy liczba wierszy się zgadza.

Pokaż wynik:
- ścieżkę do pliku i liczbę wierszy,
- potwierdzenie, że plik wczytuje się poprawnie.

Warunek poprawności:
Liczba wierszy po wczytaniu jest taka sama jak przed zapisem.

Nie rób już nic więcej.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć
- Potwierdzenie zapisu `teksty.csv`.

**Na koniec:** zapisz notatnik, a potem **Source Control → opis zmiany → Commit → Sync Changes**. Plik `data/processed/teksty.csv` trafi do repozytorium, a folder `data/raw/` zostanie tylko w Twoim codespace.

#### Pytanie interpretacyjne
Czego brakuje w zebranym materiale? Kogo nie słychać w tych tekstach? Zapisz odpowiedź w polu 10 karty projektu.